# 03 - Silver Bookings Validation Refresh

Sanitized portfolio notebook for the Corporate Travel Booking Data Platform.

Before running this notebook in a real Databricks workspace, replace placeholder paths such as `<storage-account>` and confirm that the Unity Catalog schemas exist.
All outputs have been cleared before publishing.


In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import when, col, lit, concat_ws, lower, coalesce

PROCESS_NAME = "silver_validation"
WATERMARK_TABLE = "corporate_travel_analytics.control.pipeline_watermark"
SOURCE_TABLE = "corporate_travel_analytics.bronze.bookings_clean_stage"
SILVER_CLEAN_TABLE = "corporate_travel_analytics.silver.bookings_clean"
SILVER_QUARANTINE_TABLE = "corporate_travel_analytics.silver.bookings_quarantine"

# =========================================================
# 1. READ LAST WATERMARK
# =========================================================
watermark_df = (
    spark.table(WATERMARK_TABLE)
    .filter(col("process_name") == PROCESS_NAME)
)

last_watermark = watermark_df.collect()[0]["last_ingest_time"]

print(f"Last watermark for {PROCESS_NAME}: {last_watermark}")

# =========================================================
# 2. READ ONLY CHANGED ROWS FROM BRONZE CLEAN STAGE
# =========================================================
changed_df = (
    spark.table(SOURCE_TABLE)
    .filter(col("ingest_time") > last_watermark)
)

if changed_df.limit(1).count() == 0:
    print("No changed rows found. Nothing to process.")
else:
    print(f"Changed rows found after watermark: {changed_df.count()}")

    # =========================================================
    # 3. VALIDATE ONLY CHANGED BOOKINGS
    # =========================================================
    validated_df = (
        changed_df
        .withColumn(
            "booking_id_valid",
            col("booking_id").rlike("^BT-[0-9]{4}-[0-9]{8}$")
        )
        .withColumn(
            "origin_destination_valid",
            when(
                col("origin").isNotNull() &
                col("destination").isNotNull() &
                (lower(col("origin")) != lower(col("destination"))),
                lit(True)
            ).otherwise(lit(False))
        )
        .withColumn(
            "roundtrip_return_valid",
            when(
                (col("trip_type") == "RoundTrip") & col("return_ts").isNull(),
                lit(False)
            ).otherwise(lit(True))
        )
        .withColumn(
        "cost_valid", 
        when(
            (coalesce(col("total_cost"), lit(0)) == 
             (coalesce(col("flight_cost"), lit(0)) + coalesce(col("hotel_cost"), lit(0)))) & 
            (coalesce(col("total_cost"), lit(0)) > 0), # Forces 0 values to False
            lit(True)
        ).otherwise(lit(False))
    )

        .withColumn(
            "poc_email_valid",
            col("poc_email").isNotNull()
        )
    )

    validated_df = validated_df.withColumn(
        "validation_error",
        concat_ws(
            "; ",
            when(~col("booking_id_valid"), lit("Invalid booking_id")),
            when(~col("origin_destination_valid"), lit("Origin and destination invalid or same")),
            when(~col("roundtrip_return_valid"), lit("RoundTrip missing return_ts")),
            when(~col("cost_valid"), lit("total_cost mismatch")),
            when(~col("poc_email_valid"), lit("Missing poc_email"))
        )
    )

    valid_df = validated_df.filter(
        col("booking_id_valid") &
        col("origin_destination_valid") &
        col("roundtrip_return_valid") &
        col("cost_valid") &
        col("poc_email_valid")
    )

    invalid_df = validated_df.filter(
        ~(
            col("booking_id_valid") &
            col("origin_destination_valid") &
            col("roundtrip_return_valid") &
            col("cost_valid") &
            col("poc_email_valid")
        )
    )

    # =========================================================
    # 4. CREATE TARGET TABLES IF NOT EXISTS
    # =========================================================
    if not spark.catalog.tableExists(SILVER_CLEAN_TABLE):
        valid_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(SILVER_CLEAN_TABLE)
        print(f"Created target table: {SILVER_CLEAN_TABLE}")

    if not spark.catalog.tableExists(SILVER_QUARANTINE_TABLE):
        invalid_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(SILVER_QUARANTINE_TABLE)
        print(f"Created target table: {SILVER_QUARANTINE_TABLE}")

    # =========================================================
    # 5. MERGE VALID ROWS INTO SILVER CLEAN
    # =========================================================
    if valid_df.limit(1).count() > 0:
        silver_clean_target = DeltaTable.forName(spark, SILVER_CLEAN_TABLE)

        silver_clean_target.alias("t").merge(
            valid_df.alias("s"),
            "t.booking_id = s.booking_id"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()

        print("Merged valid rows into silver.bookings_clean")

        # Remove same booking_id from quarantine if now valid
        valid_ids_df = valid_df.select("booking_id").distinct()
        valid_ids_df.createOrReplaceTempView("valid_ids_to_remove_from_quarantine")

        spark.sql(f"""
            DELETE FROM {SILVER_QUARANTINE_TABLE}
            WHERE booking_id IN (
                SELECT booking_id FROM valid_ids_to_remove_from_quarantine
            )
        """)

        print("Removed valid booking_ids from silver.bookings_quarantine")

    # =========================================================
    # 6. MERGE INVALID ROWS INTO SILVER QUARANTINE
    # =========================================================
    if invalid_df.limit(1).count() > 0:
        silver_quarantine_target = DeltaTable.forName(spark, SILVER_QUARANTINE_TABLE)

        silver_quarantine_target.alias("t").merge(
            invalid_df.alias("s"),
            "t.booking_id = s.booking_id"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()

        print("Merged invalid rows into silver.bookings_quarantine")

        # Remove same booking_id from clean if now invalid
        invalid_ids_df = invalid_df.select("booking_id").distinct()
        invalid_ids_df.createOrReplaceTempView("invalid_ids_to_remove_from_clean")

        spark.sql(f"""
            DELETE FROM {SILVER_CLEAN_TABLE}
            WHERE booking_id IN (
                SELECT booking_id FROM invalid_ids_to_remove_from_clean
            )
        """)

        print("Removed invalid booking_ids from silver.bookings_clean")

    # =========================================================
    # 7. UPDATE WATERMARK
    # =========================================================
    max_ingest_time = changed_df.agg({"ingest_time": "max"}).collect()[0][0]

    spark.sql(f"""
        DELETE FROM {WATERMARK_TABLE}
        WHERE process_name = '{PROCESS_NAME}'
    """)

    spark.sql(f"""
        INSERT INTO {WATERMARK_TABLE}
        VALUES ('{PROCESS_NAME}', TIMESTAMP('{max_ingest_time}'))
    """)

    print(f"Updated watermark for {PROCESS_NAME} to {max_ingest_time}")
